In [ ]:
#By Parisa Hajialigol - 2025

#This code is hierarchical RL-MPC with adding the Model of FSC, AbC, and the cooling storage system.
#Integrating Model Predictive Control (MPC) at the individual building level and Soft Actor-Critic (SAC) reinforcement learning at the district level within the CityLearn environment is a strategic approach to optimize both local and communal energy management objectives. Here's a structured plan to implement this hierarchical control system:
#LSTM model is trained for indoor temperature prediction
#MPC Objective: Optimize each building's energy consumption while maintaining indoor temperatures within a specified comfort range.
#RL-SAC Objective: Coordinate charging and discharging of communal energy storage systems to optimize district-wide energy efficiency and cost.​


In [ ]:
#Environment Setup
!pip install CityLearn
!pip install --upgrade stable-baselines3
!pip install --upgrade shimmy
!pip install --upgrade gymnasium

In [ ]:
!pip uninstall numpy -y  # Uninstall current numpy
!pip install numpy==1.25.0  # Install a specific numpy version

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Using cached numpy-1.25.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.6 kB)
Using cached numpy-1.25.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (17.6 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 1.25.0 which is incompatible.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.25.0 which is incompatible.
xarray 2025.7.1 requires numpy>=1.26, but you have numpy 1.25.0 which is incompatible.
yfinance 0.2.65 requires beautifulsoup4>=4.11.1, but you have beautifulsoup4 4.8.0 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.25.0 which is incompatible.
um

In [ ]:
# Import necessary libraries
import os
import shutil
import json

import torch
from torch.utils.data import TensorDataset, DataLoader
from citylearn.building import Building
from citylearn.data import get_settings
from citylearn.preprocessing import Normalize, PeriodicNormalization

import numpy as np
import pandas as pd

from gymnasium.wrappers import RecordVideo

from stable_baselines3 import SAC
from citylearn.agents.rbc import BasicRBC
from citylearn.citylearn import CityLearnEnv
from citylearn.data import DataSet
from citylearn.reward_function import RewardFunction
from citylearn.utilities import read_json, write_json
from citylearn.wrappers import NormalizedObservationWrapper, StableBaselines3Wrapper

from citylearn.end_use_load_profiles.lstm_model.model import LSTM
from citylearn.end_use_load_profiles.lstm_model.model_generation import run, eval, train, _train
from citylearn.end_use_load_profiles.lstm_model.preprocessing import preprocess_df


ModuleNotFoundError: No module named 'citylearn.utils'

New Energy System Models

In [ ]:
#Fresnel Collector Model
class FresnelCollector:
    def __init__(self, nominal_power=5.0, efficiency=0.5):
        self.nominal_power = nominal_power
        self.efficiency = efficiency
        self.collected_energy = []  # Store thermal output per timestep
        self.used_energy = []
        self.wasted_energy = []

    def get_output(self, solar_irradiance):
        output = self.efficiency * solar_irradiance * self.nominal_power

        # Ensure lists remain lists
        if not isinstance(self.collected_energy, list):
            self.collected_energy = self.collected_energy.tolist()

        self.collected_energy.append(output)
        return output

    def log_usage(self, used, total):
        if not isinstance(self.used_energy, list):
            self.used_energy = self.used_energy.tolist()
        if not isinstance(self.wasted_energy, list):
            self.wasted_energy = self.wasted_energy.tolist()

        self.used_energy.append(used)
        self.wasted_energy.append(total - used)


In [ ]:
#Thermal Buffer Model
class ThermalBuffer:
    def __init__(self, capacity_kWh=200.0, loss=0.01):
        self.capacity = capacity_kWh
        self.soc = 0.0
        self.loss = loss
        self.soc_history = []

    def charge(self, energy_kWh):
        self.soc = min(self.capacity, self.soc + energy_kWh)

    def discharge(self, requested_kWh):
        available = min(requested_kWh, self.soc)
        self.soc -= available
        return available

    def step(self):
        self.soc *= (1 - self.loss)
        self.soc_history.append(self.soc)


In [ ]:
# Absorption Chiller Model
class AbsorptionChiller:
    def __init__(self, thermal_capacity_kW=50.0, cop=0.7):
        self.thermal_capacity = thermal_capacity_kW
        self.cop = cop
        self.cooling_output_history = []

    def get_cooling_output(self, thermal_input_kWh):
        actual_input = min(thermal_input_kWh, self.thermal_capacity)
        cooling_output = self.cop * actual_input
        self.cooling_output_history.append(cooling_output)
        return cooling_output

In [ ]:
# Load Data
from google.colab import drive
drive.mount('/content/drive')

data_path = '/content/drive/MyDrive/CityLearn-master/NewData/'

schema_path = os.path.join(data_path, 'schema.json')

# Load the schema
with open(schema_path, 'r') as f:
    schema = json.load(f)

schema['root_directory'] = data_path # Make root_directory an absolute path



In [ ]:
# Instantiate CityLearnEnv
env = CityLearnEnv(data_path=data_path, schema=schema, central_agent=False)
drive.mount("/content/drive", force_remount=True)
# env = CityLearnEnv('citylearn_challenge_2023_phase_2_local_evaluation', central_agent=True)


In [ ]:
for b in env.buildings:
    if b.name == 'Building_3':
        b.solar_collector = FresnelCollector(nominal_power=184.32, efficiency=0.60)
        b.abc = AbsorptionChiller(thermal_capacity_kW=50, cop=0.7)
        b.thermal_buffer = ThermalBuffer(capacity_kWh=200)

In [ ]:
#Check
for i, b in enumerate(env.buildings):
    print(f"\n Building {i+1}: {b.name}")

    # Absorption Chiller
    if hasattr(b, 'abc'):
        print(f"  Absorption Chiller COP: {b.abc.cop}")
        print(f"  Absorption Chiller Thermal Capacity: {b.abc.thermal_capacity} kW")

    # Thermal Buffer
    if hasattr(b, 'thermal_buffer'):
        print(f"  Thermal Buffer Capacity: {getattr(b.thermal_buffer, 'capacity', 'N/A')} kWh")

    # Battery
    if hasattr(b, 'electrical_storage'):
        print(f"  Battery Storage Capacity: {getattr(b.electrical_storage, 'capacity', 'N/A')} kWh")

    # DHW
    if hasattr(b, 'dhw_storage'):
        print(f"  DHW Storage Capacity: {getattr(b.dhw_storage, 'capacity', 'N/A')} kWh")

    # PV
    if hasattr(b, 'pv'):
        print(f"  PV Nominal Power: {getattr(b.pv, 'nominal_power', 'N/A')} kW")

    # Solar Collector
    if hasattr(b, 'solar_collector'):
        print(f"  Solar Collector Nominal Power: {b.solar_collector.nominal_power} kW")

print(env.observation_names)
print(env.observation_space)


In [ ]:
# No control
from citylearn.agents.base import BaselineAgent as Agent
model = Agent(env)

observations, _ = env.reset()
nominal_power = 184.32
efficiency = 0.6

#Tracking for plots
thermal_output_history = []
fsc_used_energy_history = []
cooling_output_history = []
dhw_energy_used_history = []

while not env.terminated:
    actions = model.predict(observations[0])
    observations, reward, info, terminated, truncated = env.step(actions)
    t = env.time_step

    for b in env.buildings:
        obs = b.observations()
        irr = obs['direct_solar_irradiance'] + obs['diffuse_solar_irradiance']
        thermal_output = (efficiency * irr * nominal_power) / 1000  # kWh

        total_used = 0  # to log FSC usage

        #BUILDING 3: FSC + Buffer
        if b.name == 'Building_3' and hasattr(b, 'solar_collector'):
            # Collect energy and store in buffer
            b.thermal_buffer.charge(thermal_output)

            # DHW from buffer
            soc_before = b.dhw_storage.soc[t]
            energy_needed = b.dhw_storage.capacity * (1.0 - soc_before)
            dhw_energy = b.thermal_buffer.discharge(energy_needed)
            b.dhw_storage.charge(dhw_energy)
            soc_after = b.dhw_storage.soc[t]
            used_dhw = b.dhw_storage.capacity * (soc_after - soc_before)
            total_used += used_dhw
            dhw_energy_used_history.append(used_dhw)

            # AbC from buffer
            heat_for_abc = b.thermal_buffer.discharge(b.abc.thermal_capacity)
            cooling = b.abc.get_cooling_output(heat_for_abc)
            cooling_output_history.append(cooling)
            total_used += heat_for_abc

            # Log FSC usage
            b.solar_collector.log_usage(total_used, thermal_output)
            thermal_output_history.append(thermal_output)
            fsc_used_energy_history.append(total_used)

            # Apply loss to buffer
            b.thermal_buffer.step()

        #ALL buildings: DHW storage
        elif hasattr(b, 'dhw_storage'):
            soc_before = b.dhw_storage.soc[t]
            b.dhw_storage.charge(0)  # No FSC, but keep SOC updated
            soc_after = b.dhw_storage.soc[t]


In [ ]:
import matplotlib.pyplot as plt

timesteps = range(len(thermal_output_history))

plt.figure(figsize=(15, 10))

# FSC Energy
plt.subplot(3, 1, 1)
plt.plot(timesteps, thermal_output_history, label='FSC Collected Energy')
plt.plot(timesteps, fsc_used_energy_history, '--', label='FSC Used Energy')
plt.title('FSC Thermal Energy Flow (Building_3)')
plt.ylabel('kWh')
plt.legend()
plt.grid(True)

# Cooling Output
plt.subplot(3, 1, 2)
plt.plot(timesteps, cooling_output_history, color='blue')
plt.title('Cooling Output from Absorption Chiller (Building_3)')
plt.ylabel('kWh')
plt.grid(True)

# DHW Energy Use
plt.subplot(3, 1, 3)
plt.plot(timesteps, dhw_energy_used_history, color='orange')
plt.title('DHW Energy Supplied from Buffer (Building_3)')
plt.xlabel('Timestep')
plt.ylabel('kWh')
plt.grid(True)

plt.tight_layout()
plt.show()




In [ ]:
#Baseline (No Control)
#KPIs
base_kpis = model.env.evaluate()
base_kpis = base_kpis.pivot(index='cost_function', columns='name', values='value').round(3)
base_kpis = base_kpis.dropna(how='all')
display(base_kpis)


In [ ]:
#Check
t = 0
obs = env.reset()

if isinstance(obs, tuple):
    obs = obs[0]

print(f"\n Total Buildings: {len(env.buildings)}")
print(f" Length of observation_names: {len(env.observation_names)}")
print(f" Sample obs shape/type: {type(obs)}, length: {len(obs)}")

# Let's check each building's obs
for i, building_obs in enumerate(obs):
    print(f"\n Building {i+1} Observations:")
    try:
        for name, val in zip(env.observation_names, building_obs):
            print(f"  {name}: {val}")
    except Exception as e:
        print(f" Error with building {i+1}: {e}")


In [ ]:
#Reload Dataset
from google.colab import files
uploaded = files.upload()

In [ ]:
# Replace the file names with the correct uploaded file names
building_df = pd.read_csv('Building_1.csv')
weather_df = pd.read_csv('weather.csv')

# Merge on a common column, e.g., 'timestamp'
df = pd.merge(building_df, weather_df, left_index=True, right_index=True, how='inner')


# Preview the combined DataFrame
print(df.head())

In [ ]:
# Assume `df` contains one month's data with a 'month' column
df_copied = pd.concat([df.assign(month=m) for m in range(1, 13)], ignore_index=True)

# Verify the duplicated data
print(df_copied['month'].unique())  # Should print all 12 months

In [ ]:
#LSTM
#LSTM parameters

num_features = 13 #num_features = len(env.observation_space)
num_output = 1
num_hidden = 64
num_layers = 2
drop_prob = 0.2
weight_decay = 0.0001
device = 'cuda' if torch.cuda.is_available() else 'cpu'

#LSTM model
lstm_model = LSTM(
    n_features=num_features,
    n_output=num_output,
    seq_len=24,
    num_hidden=num_hidden,
    num_layers=num_layers,
    drop_prob=drop_prob,
    weight_decay=weight_decay
).to(device)

print("LSTM Model Initialized")

In [ ]:
#Train LSTM
config = {
    "learning_rate": 0.001,
    "epochs": 10,
    "batch_size": 32,
    "hidden_units": 64,
    "save_model_path": "my_lstm_model.pth",
    "weight_decay": 0.0001,
    "device": 'cuda' if torch.cuda.is_available() else 'cpu',
    "lb": 24
}

from citylearn.end_use_load_profiles.lstm_model.preprocessing import preprocess_df as original_preprocess_df

def my_preprocess_df(config, df, train_references=None, validation_references=None, test_references=None):
    """
    Custom preprocessing function without the 12-month assertion.
    """
    data_dict = None

    # Call the original preprocessing function, but ignore the assertion
    try:
        print("Features used for LSTM training:")
        print(df.columns)
        data_dict = original_preprocess_df(config, df)
        print(data_dict.keys()) # Print keys to help debug

    except AssertionError:
        print("AssertionError occurred. Printing 'df' head for debugging:")
        print(df.head())
        print("Creating an empty dictionary for data_dict as fallback")
        data_dict = {'loaders': {'train': None, 'val': None}, 'temp_limits': {'max': None, 'min': None}}
        pass


    return data_dict

if 'reference' not in df_copied.columns:
    df_copied['reference'] = 0
data_dict = preprocess_df(config, df_copied)

# Verify the output
if data_dict:
    print("Preprocessing completed successfully!")
    print(data_dict.keys())
else:
    print("Preprocessing failed.")


criterion = torch.nn.MSELoss()  # mean-squared error for regression

optimizer = torch.optim.Adam(lstm_model.parameters(), lr=config['learning_rate'], weight_decay=weight_decay)

lstm = train(
        lstm_model,
        data_dict['loaders']['train'],
        data_dict['loaders']['val'],
        optimizer,
        criterion,
        config,
        data_dict['temp_limits']['max'],
        data_dict['temp_limits']['min'],
    )
print("Training completed successfully!")

In [ ]:
# Model Evaluation
results = eval(
    config,
    lstm_model,
    data_dict['loaders']['test'],
    optimizer,
    temperature_normalization_maximum=data_dict['temp_limits']['max'],
    temperature_normalization_minimum=data_dict['temp_limits']['min']
)

# Extract the metrics from the returned dictionary
absolute_error = results['absolute error']
mae = results['mae']
rmse = results['rmse']

# Print the metrics
print(f"Absolute Error (Mean): {absolute_error}")
print(f"MAE: {mae}")
print(f"RMSE: {rmse}")

def my_eval(config, model, loader, optimizer, max_temp, min_temp):
    model.eval()
    predictions = []
    ground_truths = []
    total_loss = 0
    device = config.get('device', 'cpu')
    batch_size = config.get('batch_size', 32)

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.float().to(device), y_batch.float().to(device)

            # Initialize the hidden state
            hidden_cell_tuple = model.init_hidden(batch_size, device)

            # Call the model with hidden state
            y_pred, _ = model(X_batch, hidden_cell_tuple)

            predictions.append(y_pred.cpu().numpy()) # Move predictions back to CPU before converting to numpy
            ground_truths.append(y_batch.cpu().numpy()) # Move ground truths back to CPU before converting to numpy
            loss = torch.nn.functional.mse_loss(y_pred, y_batch)
            total_loss += loss.item()

    # Flatten predictions and ground truths for easier plotting
    predictions = np.concatenate(predictions)
    ground_truths = np.concatenate(ground_truths)

    return predictions, ground_truths, total_loss


predictions, ground_truths, total_loss = my_eval(
    config,
    lstm_model,
    data_dict['loaders']['test'],
    optimizer,
    data_dict['temp_limits']['max'],
    data_dict['temp_limits']['min'],
)

print(f"Total Test Loss: {total_loss}")

In [ ]:
import matplotlib.pyplot as plt

# Plot a subset of predictions vs ground truth
plt.figure(figsize=(12, 6))
plt.plot(ground_truths[:100], label="Indoor Temperature", linestyle="--", marker='o', alpha=0.7)
plt.plot(predictions[:100], label="Predictions", marker='o', alpha=0.7)
plt.title("LSTM Predictions vs Normalized Actual Indoor Temperature")
plt.xlabel("Time Step")
plt.ylabel("Value")
plt.legend()
plt.show()


In [ ]:
class CustomReward(RewardFunction):
    def __init__(self, env: CityLearnEnv, alpha=1.0, beta=0.2, gamma=1.0, delta=0.01, band=1.0):
        super().__init__(env)  # Ensure you pass 'env' to the superclass
        self.env = env  # Store 'env' as an instance attribute
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.delta = delta
        self.band = band
        self.prev_power = {b.name: 0.0 for b in env.buildings}


        print(f"Reward weights: alpha={self.alpha}, beta={self.beta}, gamma={self.gamma}, delta={self.delta}")


    def calculate(self, **kwargs):
        reward_list = []
        observations = kwargs.get('observations')

        for b in self.env.buildings:
            obs = b.observations()

            solar_gen = obs['solar_generation']
            net_import = max(b.net_electricity_consumption[-1] - solar_gen, 0.0)
            cost = b.net_electricity_consumption[-1] * obs['electricity_pricing']
            indoor_temp = obs['indoor_dry_bulb_temperature']
            setpoint = obs['indoor_dry_bulb_temperature_cooling_set_point']
            comfort_violation = max(0.0, abs(indoor_temp - setpoint) - self.band)

            ramping = abs(b.net_electricity_consumption[-1] - self.prev_power[b.name])
            self.prev_power[b.name] = b.net_electricity_consumption[-1]

            reward = (
                - self.alpha * net_import
                - self.beta * cost
                - self.gamma * comfort_violation
                - self.delta * ramping
            )
            reward_list.append(reward)

        return [sum(reward_list)] if self.env.central_agent else reward_list


In [ ]:
# env = CityLearnEnv('citylearn_challenge_2023_phase_2_local_evaluation', central_agent=False)
env = CityLearnEnv(data_path=data_path, schema=schema, central_agent=False)
# env.reward_function = CustomReward(env) # set custom reward

In [ ]:
# After loading the environment
print(env.schema['reward_function'])


In [ ]:
# Cost function for MPC -- Minimize Energy Usage from Grid and Deviation from comfort temperature

# Parameters for the MPC
N = 6  # Prediction horizon (e.g., 12 steps of 15 minutes each)
comfort_temp = 22  # Desired indoor temperature in degrees Celsius
comfort_range = [20, 24]  # Acceptable temperature range
alpha = 1  # Weight for comfort
beta = 2  # Weight for energy usage

def mpc_cost_function(control_sequence, *args):
    lstm_model, initial_state, external_inputs = args
    total_cost = 0
    current_state = np.array(initial_state)  # Ensure initial_state is a numpy array

    # Initialize the hidden state
    batch_size = 1  # For single sequence prediction
    device = torch.device('cpu')
    hidden_cell_tuple = lstm_model.init_hidden(batch_size, device)

    for i in range(len(control_sequence)):
        control = control_sequence[i]

        # Ensure current_state is a numpy array
        current_state = np.array(current_state).flatten()

        # Handle input_state and external_data
        # Modified this part:
        input_state = current_state
        external_data = np.array(external_inputs[i]).flatten()
        control = np.array([control])

        # Pad input_state to have (num_features - len(external_data) - 1) features
        input_state = np.pad(input_state, (0, num_features - len(external_data) - len(control) - len(input_state)), 'constant')

        # Combine inputs into one array
        input_data = np.concatenate([input_state, external_data, control])

        # Debugging: Print shapes for validation
        print(f"Input data length: {len(input_data)} | Expected: {num_features}")
        print(f"Input data: {input_data}")

        # Assert the correct input size
        assert len(input_data) == num_features, (
            f"Input data has {len(input_data)} features, expected {num_features}. "
            f"current_state: {current_state}, external_data: {external_data}, control: {control}"
        )

        # Convert input_data to a PyTorch tensor
        input_data = torch.tensor(input_data, dtype=torch.float32).reshape(1, 1, -1).to(device)

        # Predict next state using LSTM
        predicted_temp, hidden_cell_tuple = lstm_model(input_data, hidden_cell_tuple)

        predicted_temp = predicted_temp.item()  # Extract scalar from tensor

        # Calculate costs
        comfort_cost = alpha * max(0, predicted_temp - comfort_range[1]) ** 2 + \
                       alpha * max(0, comfort_range[0] - predicted_temp) ** 2
        energy_cost = beta * env.buildings[0].net_electricity_consumption[0] * abs(control[0]) # Access the first element

        # Debugging: Log costs
        print(f"Step {i}:")
        print(f"  Predicted Temperature: {predicted_temp}")
        print(f"  Comfort Cost: {comfort_cost}")
        print(f"  Energy Cost: {energy_cost}")
        print(f"  Total Cost (so far): {total_cost}")

        total_cost += comfort_cost + energy_cost
        current_state = predicted_temp  # Update state for next prediction

    return total_cost

In [ ]:
# Function to solve the MPC optimization
def solve_mpc(initial_state, external_inputs):
    # Initial guess for the control actions (e.g., maintain current setpoint)
    u0 = [comfort_temp] * N

    # Bounds for HVAC setpoints
    bounds = [(comfort_range[0], comfort_range[1]) for _ in range(N)]

    # Arguments to pass to the cost function
    args = (lstm_model, initial_state, external_inputs)

    # Solve the optimization problem
    result = minimize(mpc_cost_function, u0, args=args, bounds=bounds, method='SLSQP',
        options={'maxiter': 500, 'ftol': 1e-3, 'disp': True} )

    # Return the optimal control sequence
    return result.x


In [ ]:
# Get feature indices
column_names = env.observation_names[0]

try:
    indoor_temp_idx = column_names.index("indoor_dry_bulb_temperature")
    outdoor_temp_idx = column_names.index("outdoor_dry_bulb_temperature")
    setpoint_idx = column_names.index("indoor_dry_bulb_temperature_cooling_set_point")
    electricity_idx = column_names.index("net_electricity_consumption")
    solar_idx = column_names.index("solar_generation")
except ValueError as e:
    print(f" Column not found: {e}")
    raise

In [ ]:
from scipy.optimize import minimize

# Parameters
time_steps = 360  # Number of time steps to simulate (e.g., one year at hourly intervals)

time_steps_data = []
comfort_costs = []
energy_costs = []
total_costs = []

# Run the simulation loop
for t in range(time_steps):

    # Get observations from the environment

    # Check if we have exceeded the simulation limit
    if t >= len(env.buildings[0].energy_simulation.hvac_mode):
        print("Simulation reached the last available timestep. Stopping.")
        break

    observations = env.reset() if t == 0 else env.step(actions)
    print(observations[0][0])
    print(f"Time step: {t}")

    # Extract state and external inputs for Building_1
    current_temperature = observations[0][0][indoor_temp_idx]  # Indoor temperature
    outdoor_temperature = observations[0][0][outdoor_temp_idx]  # Outdoor temperature

    # Prepare inputs for MPC
    current_state = np.array([current_temperature])
    external_inputs = [np.array([observations[0][0][indoor_temp_idx], observations[0][0][outdoor_temp_idx]]) for _ in range(N)]

    # Solve MPC to find the optimal control sequence
    optimal_controls = solve_mpc(current_state, external_inputs)

    # Apply the first control action (receding horizon)
    optimal_setpoint = optimal_controls[0]

    # Format actions for CityLearn (control all buildings)
    actions = []
    for i in range(len(env.buildings)):
        if i == 0:  # This is Building_1
            actions.append([optimal_setpoint, 0, 0])  # Example: setpoint, 0 for other controls
        else:
            actions.append([0, 0, 0])  # Example: 0 for other buildings and controls

    if t >= time_steps - 1:
        print("Reached maximum simulation time. Stopping.")
        break

    # Step the environment
    try:
        results = env.step(actions)
        print(f"Step {t} - Step Result: {results}")  # Debugging output

        # Unpack results
        if isinstance(results, tuple) and len(results) >= 4:
            observations, rewards, done, info = results[:4]
        else:
            done = False  # Default if `done` is missing

    except IndexError as e:
        print(f"IndexError at step {t}: {e}. Ending simulation.")
        break

    # Check for `done` flag
    if done:
        print("Simulation completed successfully.")
        break


In [ ]:
import matplotlib.pyplot as plt

# Get building names from the environment
building_names = [b.name for b in env.buildings]  # Use building names from the environment

# Initialize storage for each building
data_per_building = {
    b_name: {'indoor_temp': [], 'outdoor_temp': [], 'setpoint': [], 'electricity': [], 'solar': []}
    for b_name in building_names  # Iterate through building names
}

# Run simulation for `time_steps`
time_steps = 100
observations = env.reset()

for t in range(time_steps):
    print(f"Step {t} - Full Observations Structure: {observations}")

    # Fix: Handle different return formats
    if isinstance(observations, tuple):
        observations_list = observations[0]  # Extract first element from tuple
    elif isinstance(observations, list):
        observations_list = observations  # It's already a list
    else:
        print(f"Skipping Step {t}: Unexpected `observations` format: {type(observations)}")
        continue  # Skip invalid data

    # Ensure observations list is correct length
    if not isinstance(observations_list, list) or len(observations_list) != len(building_names): # Use building_names for comparison
        print(f"Skipping Step {t}: Mismatch in number of buildings!")
        continue

    for i, building_name in enumerate(building_names): # Use building_names for iteration
        building_obs = observations_list[i]  # Extract per-building observations

        # Debugging: Check extracted data
        print(f"Step {t} - {building_name} Observations: {building_obs[:5]}")

        # Store data per building
        data_per_building[building_name]['indoor_temp'].append(building_obs[indoor_temp_idx])
        data_per_building[building_name]['outdoor_temp'].append(building_obs[outdoor_temp_idx])
        data_per_building[building_name]['setpoint'].append(building_obs[setpoint_idx])
        data_per_building[building_name]['electricity'].append(building_obs[electricity_idx])
        data_per_building[building_name]['solar'].append(building_obs[solar_idx])

    # Step environment
    actions = [[0, 0, 0]] * len(env.buildings)
    observations, rewards, terminated, truncated, info = env.step(actions)

    if terminated:
        break

In [ ]:
for b_name, data in data_per_building.items():
    df = pd.DataFrame({'hvac_setpoint': data['setpoint']})
    df.to_csv(f'mpc_setpoints_{b_name}.csv')

In [ ]:
print(f"Current State: {current_state}")
print(f"External Inputs: {external_inputs}")
print(f"Optimal Controls: {optimal_controls}")


In [ ]:
print(env.observation_names[0])
print(observations)


In [ ]:
# Plot Data Per Building
for building_name, data in data_per_building.items():
    plt.figure(figsize=(12, 6))

    # Plot indoor and outdoor temperatures
    plt.subplot(2, 1, 1)
    plt.plot(data['indoor_temp'], label="Indoor Temperature", linestyle="-")
    plt.plot(data['outdoor_temp'], label="Outdoor Temperature", linestyle="--")
    plt.plot(data['setpoint'], label="Cooling Set Point", linestyle="--")

    plt.xlabel("Time Step")
    plt.ylabel("Temperature (°C)")
    plt.title(f"{building_name} - Indoor & Outdoor Temperature and Cooling Set Point")
    plt.legend()

    # Plot electricity consumption, solar generation, and setpoints
    plt.subplot(2, 1, 2)
    plt.plot(data['electricity'], label="Net Electricity Consumption", linestyle="-")
    plt.plot(data['solar'], label="Solar Generation", linestyle="--")
    plt.xlabel("Time Step")
    plt.ylabel("Energy (kWh)")
    plt.title(f"{building_name} - Electricity and Solar Energy")
    plt.legend()

    plt.tight_layout()
    plt.show()


In [ ]:
# Get feature indices
column_names = env.observation_names[0]

try:
    indoor_temp_idx = column_names.index("indoor_dry_bulb_temperature")
    outdoor_temp_idx = column_names.index("outdoor_dry_bulb_temperature")
    setpoint_idx = column_names.index("indoor_dry_bulb_temperature_cooling_set_point")
    electricity_idx = column_names.index("net_electricity_consumption")
    solar_idx = column_names.index("solar_generation")
except ValueError as e:
    print(f" Column not found: {e}")
    raise

In [ ]:
#SAC_MPC
from stable_baselines3.sac import SAC as Agent
from citylearn.wrappers import NormalizedObservationWrapper, StableBaselines3Wrapper

# Load the environment centrally
env = CityLearnEnv(data_path=data_path, schema=schema, central_agent=True)
# env.reward_function = CustomReward(env) # set custom reward
env = NormalizedObservationWrapper(env)
env = StableBaselines3Wrapper(env)

mpc_setpoints_dict = {
    'Building_1': pd.read_csv('mpc_setpoints_Building_1.csv')['hvac_setpoint'].values,
    'Building_2': pd.read_csv('mpc_setpoints_Building_2.csv')['hvac_setpoint'].values,
    'Building_3': pd.read_csv('mpc_setpoints_Building_3.csv')['hvac_setpoint'].values,
}

sac_model = Agent('MlpPolicy', env)

# Train SAC
episodes = 2
sac_model.learn(total_timesteps=env.unwrapped.time_steps*episodes)


In [ ]:
print(env.observation_space)
print(env.unwrapped.observation_space)


In [ ]:
# Reset the environment
observations, _ = env.reset()

def scale_sac_action(raw_action, scale=0.25):
    # Clamp raw_action to [0.0, 1.0] after scaling (safety net)
    return float(np.clip(raw_action * scale, 0.0, 1.0))

while not env.unwrapped.terminated:
    sac_actions, _ = sac_model.predict(observations, deterministic=True)
    final_actions = []

    for i, building in enumerate(env.unwrapped.buildings):
        bobs = building.observations()
        indoor_temp = bobs['indoor_dry_bulb_temperature']
        outdoor_temp = bobs['outdoor_dry_bulb_temperature']
        mpc_raw = mpc_setpoints_dict[building.name][min(env.unwrapped.time_step, len(mpc_setpoints_dict[building.name]) - 1)]

        if building.name == 'Building_1':
            hvac_setpoint = max(mpc_raw, indoor_temp - 2.0)  # stricter limit
        else:
            hvac_setpoint = max(mpc_raw, indoor_temp - 3.0)

        hvac_setpoint = np.clip(hvac_setpoint, 18.0, indoor_temp - 1.5)

        # Scale SAC actions and clamp again just in case
        battery_action = np.clip(scale_sac_action(sac_actions[i * 3], scale=0.25), 0.0, 1.0)
        dhw_action = np.clip(scale_sac_action(sac_actions[i * 3 + 1], scale=0.25), 0.0, 1.0)

        final_actions.extend([hvac_setpoint, battery_action, dhw_action])

    try:
        observations, _, done, _, _ = env.step(final_actions)
    except AssertionError as e:
        print(f"AssertionError at timestep {env.unwrapped.time_step}: {e}")
        print("Using fallback actions (safe HVAC only) for this step...")

        # Fallback: only HVAC setpoints, zero battery/DHW
        final_actions = []
        for i, building in enumerate(env.unwrapped.buildings):
            bobs = building.observations()
            indoor_temp = bobs['indoor_dry_bulb_temperature']
            mpc_raw = mpc_setpoints_dict[building.name][min(env.unwrapped.time_step, len(mpc_setpoints_dict[building.name]) - 1)]
            hvac_setpoint = min(mpc_raw, indoor_temp - 0.5)
            hvac_setpoint = np.clip(hvac_setpoint, 18.0, indoor_temp - 0.5)
            final_actions.extend([hvac_setpoint, 0.0, 0.0])

        observations, _, done, _, _ = env.step(final_actions)

    if done:
        print("RL-MPC Simulation Finished")
        break


In [ ]:
#MPC Output
import pandas as pd

# Load CSV files
mpc_b1 = pd.read_csv('mpc_setpoints_Building_1.csv')
mpc_b2 = pd.read_csv('mpc_setpoints_Building_2.csv')
mpc_b3 = pd.read_csv('mpc_setpoints_Building_3.csv')

# Check statistics and validity
print("Building 1 Setpoints:")
print(mpc_b1.describe())
print(mpc_b1.head(20))

print("\nBuilding 2 Setpoints:")
print(mpc_b2.describe())
print(mpc_b2.head(20))

print("\nBuilding 3 Setpoints:")
print(mpc_b3.describe())
print(mpc_b3.head(20))


In [ ]:
# SAC Actions
sac_actions, _ = sac_model.predict(observations, deterministic=True)
print("SAC Actions:", sac_actions)
print("Min SAC Action:", sac_actions.min(), "Max SAC Action:", sac_actions.max())


In [ ]:
print(f"[STEP {env.unwrapped.time_step}] Building {building.name}")
print(f"Indoor Temp : {indoor_temp:.2f}")
print(f"Outdoor Temp: {outdoor_temp:.2f}")
print(f"Final HVAC Setpoint after clamps : {hvac_setpoint:.2f}")
print("===")


In [ ]:
print("Number of SAC actions:", len(sac_actions))
print("Number of buildings:", len(env.unwrapped.buildings))
print("Actions per building:", len(sac_actions) / len(env.unwrapped.buildings))


In [ ]:
print(f"[Step {env.unwrapped.time_step}] Final Actions: {final_actions}")


In [ ]:
# RL-MPC KPIs
mpc_kpis = env.unwrapped.evaluate()
mpc_kpis = mpc_kpis.pivot(index='cost_function', columns='name', values='value').round(3)
mpc_kpis = mpc_kpis.dropna(how='all')
display(mpc_kpis)

In [ ]:
#SAC
from stable_baselines3.sac import SAC as Agent
from citylearn.wrappers import NormalizedObservationWrapper, StableBaselines3Wrapper

env = CityLearnEnv(data_path=data_path, schema=schema, central_agent=True)
# env.reward_function = CustomReward(env) # set custom reward
env = NormalizedObservationWrapper(env)
env = StableBaselines3Wrapper(env)
sac_model = Agent('MlpPolicy', env)

# train
episodes = 2
sac_model.learn(total_timesteps=env.unwrapped.time_steps*episodes)

# test
observations, _ = env.reset()

while not env.unwrapped.terminated:
    actions, _ = sac_model.predict(observations, deterministic=True)
    observations, _, _, _, _ = env.step(actions)





In [ ]:
# SAC KPIs
sac_kpis = env.unwrapped.evaluate()
sac_kpis = sac_kpis.pivot(index='cost_function', columns='name', values='value').round(3)
sac_kpis = sac_kpis.dropna(how='all')
display(sac_kpis)


In [ ]:
#KPI Evaluation
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

#KPI Names
kpi_labels = {
    'electricity_consumption_total': 'Consumption',
    'zero_net_energy':'Zero Energy',
    'cost_total': 'Cost',
    'daily_peak_average': 'Daily peak',
    'ramping_average': 'Ramping',
    'monthly_one_minus_load_factor_average': '1 - Load factor',
    'discomfort_proportion': 'Discomfort'
    }

kpis_to_plot = list(kpi_labels.keys())

# Extract District KPIs
base_kpis_dict = base_kpis['District'].to_dict()
sac_kpis_dict = sac_kpis['District'].to_dict()
mpc_kpis_dict = mpc_kpis['District'].to_dict()

# Create DataFrame for Plot
def create_kpi_df(agent_name, kpi_dict, baseline_dict):
    data = []
    for kpi in kpis_to_plot:
        baseline_value = baseline_dict.get(kpi, None)
        value = kpi_dict.get(kpi, None)

        if value is not None and baseline_value and baseline_value != 0:
            normalized = value / baseline_value
        else:
            normalized = float('nan')

        data.append({
            'cost_function': kpi,
            'label': kpi_labels.get(kpi, kpi),
            'value': normalized,
            'agent': agent_name
        })

    return pd.DataFrame(data)

df_baseline = create_kpi_df('Baseline', base_kpis_dict, base_kpis_dict)
df_rlc = create_kpi_df('RLC', sac_kpis_dict, base_kpis_dict)
df_mpc = create_kpi_df('RL-MPC', mpc_kpis_dict, base_kpis_dict)



data = pd.concat([df_rlc, df_mpc], ignore_index=True)



In [ ]:
import pandas as pd

# Select District KPIs only
kpis_to_show = [
    'electricity_consumption_total',
    'zero_net_energy',
    'cost_total',
    'daily_peak_average',
    'ramping_average',
    'monthly_one_minus_load_factor_average',
    'discomfort_proportion'
    ]

# Friendly Names
kpi_labels = {
    'electricity_consumption_total': 'Consumption',
    'zero_net_energy':'Zero Energy',
    'cost_total': 'Cost',
    'daily_peak_average': 'Daily peak',
    'ramping_average': 'Ramping',
    'monthly_one_minus_load_factor_average': '1 - Load factor',
    'discomfort_proportion': 'Discomfort'
    }

# Create Table
final_kpis = pd.DataFrame({
    'KPI': [kpi_labels[k] for k in kpis_to_show],
    'Baseline': [base_kpis['District'][k] for k in kpis_to_show],
    'RLC (SAC)': [sac_kpis['District'][k] for k in kpis_to_show],
    'RL-MPC': [mpc_kpis['District'][k] for k in kpis_to_show],
})

# Round values
final_kpis = final_kpis.round(3)

# Show Table
display(final_kpis)


In [ ]:
#KPI barplot
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(
    x='value',
    y='label',
    hue='agent',
    data=data,
    order=kpi_labels.values(),
    hue_order=[ 'RLC', 'RL-MPC'],
    ax=ax
)

# Plot a vertical line at 1.0 (Baseline reference)
ax.axvline(1.0, color='black', linestyle='--', label='Baseline Reference')

# Labels & aesthetics
ax.set_xlabel(r'$\frac{KPI}{KPI_{Baseline}}$')
ax.set_ylabel(None)
ax.set_title("Normalized KPI Comparison")

for s in ['right', 'top']:
    ax.spines[s].set_visible(False)

# Annotate bars
for p in ax.patches:
    value = p.get_width()
    if not pd.isna(value):
        ax.text(
            p.get_x() + value + 0.01,
            p.get_y() + p.get_height() / 2,
            f'{value:.2f}',
            va='center',
            ha='left',
            fontsize=8
        )

# Legend
ax.legend(
    bbox_to_anchor=(1.02, 1.0),
    loc='upper left',
    framealpha=0
)

plt.tight_layout()
plt.show()


In [ ]:
df = pd.read_csv("mpc_setpoints_Building_1.csv")
print(df['hvac_setpoint'].describe())